# Silver transformations

Reads the Bronze Parquet landing area, validates and deduplicates source records, then builds/merges Delta tables in `adb_adventureworks_dev.silver`. Run after the Bronze ADF loads and Bronze validation.

In [ ]:
print("Spark version:", spark.version)

print("Databricks connection successful.")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_path = "abfss://bronze@awdata2022.dfs.core.windows.net/"

customer_full_path = bronze_path + "sales/customer/customer.parquet"
customer_incremental_path = bronze_path + "sales/customer/incremental/"

customer_full_df = spark.read.parquet(customer_full_path)
customer_incremental_df = spark.read.parquet(customer_incremental_path)
small_incremental_df = customer_incremental_df
customer_full_1 = customer_full_df
customer_combined = customer_full_df.unionByName(customer_incremental_df)

print("Customer baseline rows:", customer_full_df.count())
print("Customer incremental rows:", customer_incremental_df.count())


In [ ]:
incremental_comparison = (
    small_incremental_df.alias("inc")
    .join(
        customer_full_1.select("CustomerID", "ModifiedDate").alias("base"),
        on="CustomerID",
        how="left"
    )
    .select(
        "CustomerID",
        F.col("inc.ModifiedDate").alias("IncrementalModifiedDate"),
        F.col("base.ModifiedDate").alias("BaselineModifiedDate"),
        F.when(
            F.col("base.CustomerID").isNull(),
            "New customer"
        ).when(
            F.col("inc.ModifiedDate") > F.col("base.ModifiedDate"),
            "Updated customer"
        ).otherwise(
            "Same or older version"
        ).alias("ChangeType")
    )
    .orderBy("CustomerID")
)

display(incremental_comparison)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

customer_window = (
    Window
    .partitionBy("CustomerID")
    .orderBy(F.col("ModifiedDate").desc())
)

deduplicated_incremental = (
    small_incremental_df
    .withColumn(
        "row_number",
        F.row_number().over(customer_window)
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print("Original incremental rows:", small_incremental_df.count())
print("Rows after deduplication:", deduplicated_incremental.count())

display(
    deduplicated_incremental
    .select("CustomerID", "AccountNumber", "ModifiedDate")
    .orderBy("CustomerID")
)

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

merge_window = (
    Window
    .partitionBy("CustomerID")
    .orderBy(F.col("ModifiedDate").desc())
)

customer_latest = (
    customer_combined
    .withColumn(
        "row_number",
        F.row_number().over(merge_window)
    )
    .filter(F.col("row_number") == 1)
    .drop("row_number")
)

print("Rows after deduplication:", customer_latest.count())

print(
    "Distinct CustomerIDs:",
    customer_latest.select("CustomerID").distinct().count()
)

display(
    customer_latest
    .filter(F.col("CustomerID").isin(1, 2, 3, 4))
    .select("CustomerID", "AccountNumber", "ModifiedDate")
    .orderBy("CustomerID")
)

In [ ]:
print("Total Silver-ready rows:", customer_latest.count())

print(
    "Distinct CustomerIDs:",
    customer_latest.select("CustomerID").distinct().count()
)

print(
    "Duplicate CustomerIDs:",
    customer_latest.groupBy("CustomerID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

In [ ]:
print("Current catalog:", spark.catalog.currentCatalog())
print("Current schema:", spark.catalog.currentDatabase())

display(spark.sql("SHOW CATALOGS"))

In [ ]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS adb_adventureworks_dev.silver
COMMENT 'Cleaned and validated Silver-layer tables for the AdventureWorks data platform'
""")

print("Silver schema created successfully.")

In [ ]:
display(
    spark.sql(
        "SHOW SCHEMAS IN adb_adventureworks_dev"
    )
)

In [ ]:
display(
    spark.sql(
        "SHOW TABLES IN adb_adventureworks_dev.silver"
    )
)

In [ ]:
silver_table = "adb_adventureworks_dev.silver.customer"

customer_latest.createOrReplaceTempView("customer_silver_stage")

spark.sql("""
CREATE OR REPLACE TABLE adb_adventureworks_dev.silver.customer
USING DELTA
AS
SELECT *
FROM customer_silver_stage
""")

print(f"Created Silver table: {silver_table}")

In [ ]:
silver_customer = spark.table(
    "adb_adventureworks_dev.silver.customer"
)

print("Total rows:", silver_customer.count())

print(
    "Distinct CustomerIDs:",
    silver_customer.select("CustomerID").distinct().count()
)

print(
    "Duplicate CustomerIDs:",
    silver_customer.groupBy("CustomerID")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

display(silver_customer.limit(10))

In [ ]:
customer_type_check = silver_customer.agg(
    F.count("*").alias("TotalCustomers"),

    F.sum(
        F.when(
            F.col("PersonID").isNotNull()
            & F.col("StoreID").isNull(), 1
        ).otherwise(0)
    ).alias("IndividualCustomers"),

    F.sum(
        F.when(
            F.col("PersonID").isNull()
            & F.col("StoreID").isNotNull(), 1
        ).otherwise(0)
    ).alias("StoreCustomers"),

    F.sum(
        F.when(
            F.col("PersonID").isNull()
            & F.col("StoreID").isNull(), 1
        ).otherwise(0)
    ).alias("MissingBoth"),

    F.sum(
        F.when(
            F.col("PersonID").isNotNull()
            & F.col("StoreID").isNotNull(), 1
        ).otherwise(0)
    ).alias("PopulatedBoth")
)

display(customer_type_check)

In [ ]:
display(
    silver_customer
    .filter(
        F.col("PersonID").isNotNull()
        & F.col("StoreID").isNotNull()
    )
    .select(
        "CustomerID",
        "PersonID",
        "StoreID",
        "TerritoryID",
        "AccountNumber"
    )
    .orderBy("CustomerID")
    .limit(15)
)

In [ ]:
display(
    spark.sql("""
        DESCRIBE HISTORY adb_adventureworks_dev.silver.customer
    """)
)

In [ ]:
salesorderheader_path = (
    bronze_path + "sales/salesorderheader/"
)

display(
    dbutils.fs.ls(salesorderheader_path)
)

In [ ]:
salesorderheader_incremental_path = (
    salesorderheader_path + "incremental/"
)

display(
    dbutils.fs.ls(salesorderheader_incremental_path)
)

In [ ]:
from pyspark.sql import functions as F

salesorderheader_incremental_df = spark.read.parquet(
    salesorderheader_incremental_path
)

header_file_summary = (
    salesorderheader_incremental_df
    .withColumn(
        "SourceFile",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"([^/]+)$",
            1
        )
    )
    .groupBy("SourceFile")
    .agg(
        F.count("*").alias("RowCount"),
        F.min("ModifiedDate").alias("OldestModifiedDate"),
        F.max("ModifiedDate").alias("LatestModifiedDate")
    )
    .orderBy("SourceFile")
)

display(header_file_summary)

In [ ]:
from pyspark.sql import functions as F

salesorderheader_incremental_df = spark.read.parquet(
    salesorderheader_incremental_path
)

incremental_check = salesorderheader_incremental_df.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("SalesOrderID").alias("DistinctSalesOrderIDs"),
    F.countDistinct(
        F.struct(*salesorderheader_incremental_df.columns)
    ).alias("DistinctFullRecords"),
    F.min("ModifiedDate").alias("OldestModifiedDate"),
    F.max("ModifiedDate").alias("LatestModifiedDate")
)

display(incremental_check)

In [ ]:
salesorderheader_full_path = salesorderheader_path + "SalesOrderHeader.parquet"

salesorderheader_full_df = spark.read.parquet(
    salesorderheader_full_path
)

salesorderheader_full_df.printSchema()

header_baseline_check = salesorderheader_full_df.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("SalesOrderID").alias("DistinctSalesOrderIDs"),
    F.sum(
        F.when(F.col("SalesOrderID").isNull(), 1).otherwise(0)
    ).alias("NullSalesOrderIDs"),
    F.min("ModifiedDate").alias("OldestModifiedDate"),
    F.max("ModifiedDate").alias("LatestModifiedDate")
)

display(header_baseline_check)


In [ ]:
display(header_baseline_check)

In [ ]:
# Remove exact duplicates from the incremental extracts.
header_incremental_distinct = (
    salesorderheader_incremental_df
    .dropDuplicates()
)

# Get the matching records from the full-load baseline.
header_baseline_matches = (
    salesorderheader_full_df
    .join(
        header_incremental_distinct.select("SalesOrderID"),
        on="SalesOrderID",
        how="inner"
    )
    .select(*salesorderheader_full_df.columns)
)

# Compare every column in both directions.
incremental_not_in_baseline = (
    header_incremental_distinct
    .exceptAll(header_baseline_matches)
)

baseline_not_in_incremental = (
    header_baseline_matches
    .exceptAll(header_incremental_distinct)
)

comparison_results = [
    ("Distinct incremental records", header_incremental_distinct.count()),
    ("Matching baseline records", header_baseline_matches.count()),
    ("Incremental records differing from baseline",
     incremental_not_in_baseline.count()),
    ("Baseline records differing from incremental",
     baseline_not_in_incremental.count())
]

display(
    spark.createDataFrame(
        comparison_results,
        ["Check", "RowCount"]
    )
)

In [ ]:
from pyspark.sql import functions as F

header_quality_check = salesorderheader_full_df.agg(
    F.sum(F.when(F.col("SalesOrderNumber").isNull(), 1).otherwise(0)).alias("NullSalesOrderNumber"),
    F.sum(F.when(F.col("OrderDate").isNull(), 1).otherwise(0)).alias("NullOrderDate"),
    F.sum(F.when(F.col("CustomerID").isNull(), 1).otherwise(0)).alias("NullCustomerID"),
    F.sum(F.when(F.col("TotalDue").isNull(), 1).otherwise(0)).alias("NullTotalDue"),
    F.sum(F.when(F.col("Status").isNull(), 1).otherwise(0)).alias("NullStatus"),
    F.sum(F.when(F.col("SalesOrderID") <= 0, 1).otherwise(0)).alias("InvalidSalesOrderID"),
    F.sum(F.when(~F.col("Status").between(1, 6), 1).otherwise(0)).alias("InvalidStatus"),
    F.sum(F.when(F.col("DueDate") < F.col("OrderDate"), 1).otherwise(0)).alias("DueDateBeforeOrderDate"),
    F.sum(
        F.when(
            F.abs(
                F.col("TotalDue") -
                (F.col("SubTotal") + F.col("TaxAmt") + F.col("Freight"))
            ) > F.lit(0.0001),
            1
        ).otherwise(0)
    ).alias("MonetaryTotalMismatches")
)

display(header_quality_check)

In [ ]:
silver_customer = spark.table(
    "adb_adventureworks_dev.silver.customer"
)

header_customer_check = (
    salesorderheader_full_df
    .join(
        silver_customer.select("CustomerID"),
        on="CustomerID",
        how="left_anti"
    )
)

display(
    header_customer_check.select(
        "SalesOrderID",
        "CustomerID"
    )
)

print(
    "Orders with no matching Silver customer:",
    header_customer_check.count()
)

In [ ]:
salesorderheader_full_df.createOrReplaceTempView(
    "salesorderheader_silver_stage"
)

spark.sql("""
CREATE OR REPLACE TABLE adb_adventureworks_dev.silver.salesorderheader
USING DELTA
AS
SELECT *
FROM salesorderheader_silver_stage
""")

In [ ]:
silver_salesorderheader = spark.table(
    "adb_adventureworks_dev.silver.salesorderheader"
)

header_silver_verification = silver_salesorderheader.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("SalesOrderID").alias("DistinctSalesOrderIDs"),
    F.sum(
        F.when(F.col("SalesOrderID").isNull(), 1).otherwise(0)
    ).alias("NullSalesOrderIDs"),
    F.min("ModifiedDate").alias("OldestModifiedDate"),
    F.max("ModifiedDate").alias("LatestModifiedDate")
)

display(header_silver_verification)

In [ ]:
salesorderdetail_path = (
    bronze_path + "sales/salesorderdetail/"
)

display(dbutils.fs.ls(salesorderdetail_path))

In [ ]:
salesorderdetail_incremental_path = (
    salesorderdetail_path + "incremental/"
)

display(dbutils.fs.ls(salesorderdetail_incremental_path))

In [ ]:
from pyspark.sql import functions as F

salesorderdetail_incremental_df = spark.read.parquet(
    salesorderdetail_incremental_path
)

detail_file_summary = (
    salesorderdetail_incremental_df
    .withColumn(
        "SourceFile",
        F.regexp_extract(
            F.col("_metadata.file_path"),
            r"([^/]+)$",
            1
        )
    )
    .groupBy("SourceFile")
    .agg(
        F.count("*").alias("RowCount"),
        F.min("ModifiedDate").alias("OldestModifiedDate"),
        F.max("ModifiedDate").alias("LatestModifiedDate")
    )
    .orderBy("SourceFile")
)

display(detail_file_summary)

In [ ]:
detail_incremental_check = salesorderdetail_incremental_df.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct(
        F.struct("SalesOrderID", "SalesOrderDetailID")
    ).alias("DistinctCompositeKeys"),
    F.countDistinct(
        F.struct(*salesorderdetail_incremental_df.columns)
    ).alias("DistinctFullRecords"),
    F.sum(
        F.when(
            F.col("SalesOrderID").isNull() |
            F.col("SalesOrderDetailID").isNull(),
            1
        ).otherwise(0)
    ).alias("RowsWithNullKeyFields")
)

display(detail_incremental_check)

In [ ]:
salesorderdetail_full_path = (
    salesorderdetail_path + "SalesOrderDetail.parquet"
)

salesorderdetail_full_df = spark.read.parquet(
    salesorderdetail_full_path
)

salesorderdetail_full_df.printSchema()

detail_baseline_check = salesorderdetail_full_df.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct(
        F.struct("SalesOrderID", "SalesOrderDetailID")
    ).alias("DistinctCompositeKeys"),
    F.sum(
        F.when(
            F.col("SalesOrderID").isNull() |
            F.col("SalesOrderDetailID").isNull(),
            1
        ).otherwise(0)
    ).alias("RowsWithNullKeyFields"),
    F.min("ModifiedDate").alias("OldestModifiedDate"),
    F.max("ModifiedDate").alias("LatestModifiedDate")
)

display(detail_baseline_check)

In [ ]:
# Remove repeated copies across the six incremental files.
detail_incremental_distinct = (
    salesorderdetail_incremental_df
    .dropDuplicates()
)

# Find the matching composite keys in the full-load baseline.
detail_baseline_matches = (
    salesorderdetail_full_df
    .join(
        detail_incremental_distinct.select(
            "SalesOrderID",
            "SalesOrderDetailID"
        ),
        on=["SalesOrderID", "SalesOrderDetailID"],
        how="inner"
    )
    .select(*salesorderdetail_full_df.columns)
)

# Compare all columns in both directions.
detail_incremental_not_in_baseline = (
    detail_incremental_distinct
    .exceptAll(detail_baseline_matches)
)

detail_baseline_not_in_incremental = (
    detail_baseline_matches
    .exceptAll(detail_incremental_distinct)
)

detail_comparison_results = [
    ("Distinct incremental records", detail_incremental_distinct.count()),
    ("Matching baseline records", detail_baseline_matches.count()),
    (
        "Incremental records differing from baseline",
        detail_incremental_not_in_baseline.count()
    ),
    (
        "Baseline records differing from incremental",
        detail_baseline_not_in_incremental.count()
    )
]

display(
    spark.createDataFrame(
        detail_comparison_results,
        ["Check", "RowCount"]
    )
)

In [ ]:
detail_quality_check = salesorderdetail_full_df.agg(
    F.count("*").alias("TotalRows"),

    F.sum(F.when(F.col("OrderQty").isNull(), 1).otherwise(0))
        .alias("NullOrderQty"),

    F.sum(F.when(F.col("ProductID").isNull(), 1).otherwise(0))
        .alias("NullProductID"),

    F.sum(F.when(F.col("UnitPrice").isNull(), 1).otherwise(0))
        .alias("NullUnitPrice"),

    F.sum(F.when(F.col("UnitPriceDiscount").isNull(), 1).otherwise(0))
        .alias("NullUnitPriceDiscount"),

    F.sum(F.when(F.col("LineTotal").isNull(), 1).otherwise(0))
        .alias("NullLineTotal"),

    F.sum(F.when(F.col("OrderQty") <= 0, 1).otherwise(0))
        .alias("InvalidOrderQty"),

    F.sum(F.when(F.col("ProductID") <= 0, 1).otherwise(0))
        .alias("InvalidProductID"),

    F.sum(F.when(F.col("UnitPrice") < 0, 1).otherwise(0))
        .alias("NegativeUnitPrice"),

    F.sum(
        F.when(
            (F.col("UnitPriceDiscount") < 0) |
            (F.col("UnitPriceDiscount") > 1),
            1
        ).otherwise(0)
    ).alias("InvalidDiscount"),

    F.sum(
        F.when(
            F.abs(
                F.col("LineTotal") -
                (
                    F.col("UnitPrice") *
                    (1 - F.col("UnitPriceDiscount")) *
                    F.col("OrderQty")
                )
            ) > F.lit(0.000001),
            1
        ).otherwise(0)
    ).alias("LineTotalMismatches")
)

display(detail_quality_check)

In [ ]:
detail_order_check = (
    salesorderdetail_full_df
    .join(
        silver_salesorderheader.select("SalesOrderID"),
        on="SalesOrderID",
        how="left_anti"
    )
)

display(
    detail_order_check.select(
        "SalesOrderID",
        "SalesOrderDetailID"
    )
)

print(
    "Detail lines without a matching sales order:",
    detail_order_check.count()
)

In [ ]:
salesorderdetail_full_df.createOrReplaceTempView(
    "salesorderdetail_silver_stage"
)

spark.sql("""
CREATE OR REPLACE TABLE adb_adventureworks_dev.silver.salesorderdetail
USING DELTA
AS
SELECT *
FROM salesorderdetail_silver_stage
""")

In [ ]:
silver_salesorderdetail = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

detail_silver_verification = silver_salesorderdetail.agg(
    F.count("*").alias("TotalRows"),

    F.countDistinct(
        F.struct("SalesOrderID", "SalesOrderDetailID")
    ).alias("DistinctCompositeKeys"),

    F.sum(
        F.when(
            F.col("SalesOrderID").isNull() |
            F.col("SalesOrderDetailID").isNull(),
            1
        ).otherwise(0)
    ).alias("RowsWithNullKeyFields"),

    F.min("ModifiedDate").alias("OldestModifiedDate"),
    F.max("ModifiedDate").alias("LatestModifiedDate")
)

display(detail_silver_verification)

In [ ]:
silver_tables = [
    "adb_adventureworks_dev.silver.customer",
    "adb_adventureworks_dev.silver.salesorderheader",
    "adb_adventureworks_dev.silver.salesorderdetail"
]

for table_name in silver_tables:
    print(f"\nTable: {table_name}")

    table_df = spark.table(table_name)

    print("Rows:", table_df.count())
    print("Columns:", len(table_df.columns))

    display(
        spark.sql(f"DESCRIBE HISTORY {table_name}")
        .select("version", "timestamp", "operation", "operationMetrics")
        .limit(1)
    )

In [ ]:
product_path = (
    bronze_path + "production/product/Product.parquet"
)

product_bronze_df = spark.read.parquet(product_path)

product_bronze_df.printSchema()

print("Product Bronze row count:", product_bronze_df.count())

display(product_bronze_df.limit(10))

In [ ]:
from pyspark.sql import functions as F

product_quality_check = product_bronze_df.agg(
    F.count("*").alias("TotalRows"),

    F.countDistinct("ProductID").alias("DistinctProductIDs"),

    F.sum(
        F.when(F.col("ProductID").isNull(), 1).otherwise(0)
    ).alias("NullProductIDs"),

    F.sum(
        F.when(F.col("Name").isNull(), 1).otherwise(0)
    ).alias("NullNames"),

    F.sum(
        F.when(F.col("ProductNumber").isNull(), 1).otherwise(0)
    ).alias("NullProductNumbers"),

    F.sum(
        F.when(F.col("StandardCost") < 0, 1).otherwise(0)
    ).alias("NegativeStandardCosts"),

    F.sum(
        F.when(F.col("ListPrice") < 0, 1).otherwise(0)
    ).alias("NegativeListPrices"),

    F.sum(
        F.when(F.col("SafetyStockLevel") < 0, 1).otherwise(0)
    ).alias("NegativeSafetyStock"),

    F.sum(
        F.when(F.col("ReorderPoint") < 0, 1).otherwise(0)
    ).alias("NegativeReorderPoints"),

    F.sum(
        F.when(F.col("DaysToManufacture") < 0, 1).otherwise(0)
    ).alias("NegativeManufacturingDays")
)

display(product_quality_check)

In [ ]:
product_key_check = product_bronze_df.agg(
    F.countDistinct("ProductNumber").alias("DistinctProductNumbers"),
    F.sum(
        F.when(
            F.col("SellEndDate").isNotNull()
            & (F.col("SellEndDate") < F.col("SellStartDate")),
            1
        ).otherwise(0)
    ).alias("SellEndBeforeSellStart")
)

display(product_key_check)

In [ ]:
silver_salesorderdetail = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

unmatched_products = silver_salesorderdetail.join(
    product_bronze_df.select("ProductID"),
    on="ProductID",
    how="left_anti"
)

print("Unmatched Product references:", unmatched_products.count())

In [ ]:
product_bronze_df.createOrReplaceTempView("product_silver_stage")

spark.sql("""
CREATE OR REPLACE TABLE adb_adventureworks_dev.silver.product
USING DELTA
AS
SELECT *
FROM product_silver_stage
""")

In [ ]:
silver_product = spark.table(
    "adb_adventureworks_dev.silver.product"
)

silver_product_check = silver_product.agg(
    F.count("*").alias("TotalRows"),
    F.countDistinct("ProductID").alias("DistinctProductIDs"),
    F.countDistinct("ProductNumber").alias("DistinctProductNumbers"),
    F.sum(F.when(F.col("ProductID").isNull(), 1).otherwise(0)).alias("NullProductIDs"),
    F.sum(F.when(F.col("ProductNumber").isNull(), 1).otherwise(0)).alias("NullProductNumbers")
)

display(silver_product_check)

In [ ]:
unmatched_silver_products = (
    spark.table("adb_adventureworks_dev.silver.salesorderdetail")
    .join(
        silver_product.select("ProductID"),
        on="ProductID",
        how="left_anti"
    )
)

print(
    "Unmatched Silver Product references:",
    unmatched_silver_products.count()
)

In [ ]:
product_incremental_path = (
    bronze_path + "production/product/incremental/"
)

product_incremental_df = spark.read.parquet(
    product_incremental_path
)

print("Incremental rows:", product_incremental_df.count())

display(product_incremental_df)

In [ ]:
product_incremental_new_records = (
    product_incremental_df.exceptAll(silver_product)
)

product_incremental_new_records_count = (
    product_incremental_new_records.count()
)

print(
    "Incremental records not already present in Silver:",
    product_incremental_new_records_count
)

display(product_incremental_new_records)

In [ ]:
display(
    spark.sql("""
        DESCRIBE DETAIL adb_adventureworks_dev.silver.product
    """)
)

display(
    spark.sql("""
        DESCRIBE HISTORY adb_adventureworks_dev.silver.product
    """)
)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

product_merge_source = product_incremental_df

duplicate_product_ids = (
    product_merge_source
    .groupBy("ProductID")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate ProductIDs:", duplicate_product_ids.count())
print("Source rows:", product_merge_source.count())

In [ ]:
product_merge_comparison = (
    product_merge_source.alias("source")
    .join(
        silver_product.alias("target"),
        on="ProductID",
        how="inner"
    )
    .select(
        F.col("source.ProductID"),
        F.col("source.Name").alias("SourceName"),
        F.col("target.Name").alias("TargetName"),
        F.col("source.ModifiedDate").alias("SourceModifiedDate"),
        F.col("target.ModifiedDate").alias("TargetModifiedDate")
    )
)

display(product_merge_comparison)

In [ ]:
from delta.tables import DeltaTable

product_delta_table = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.product"
)

(
    product_delta_table.alias("target")
    .merge(
        product_merge_source.alias("source"),
        "target.ProductID = source.ProductID"
    )
    .whenMatchedUpdateAll(
        condition="source.ModifiedDate > target.ModifiedDate"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [ ]:
display(
    spark.sql("""
        DESCRIBE HISTORY adb_adventureworks_dev.silver.product
        LIMIT 1
    """).select(
        "version",
        "operation",
        "operationMetrics"
    )
)

In [ ]:
spark.sql("""
CREATE OR REPLACE TABLE adb_adventureworks_dev.silver.product_merge_test
USING DELTA
AS
SELECT *
FROM adb_adventureworks_dev.silver.product
""")

In [ ]:
product_merge_test = spark.table(
    "adb_adventureworks_dev.silver.product_merge_test"
)

print("Test table rows:", product_merge_test.count())
print(
    "Distinct ProductIDs:",
    product_merge_test.select("ProductID").distinct().count()
)

In [ ]:
from datetime import datetime
from decimal import Decimal

product_merge_test_source = spark.createDataFrame(
    [
        (
            940,
            "HL Road Pedal Updated",
            "PD-R853",
            False,
            True,
            "Silver/Black",
            500,
            375,
            Decimal("35.9596"),
            Decimal("80.9900"),
            None,
            None,
            "G",
            Decimal("149.00"),
            1,
            "R",
            "H",
            None,
            13,
            70,
            datetime(2013, 5, 30),
            None,
            None,
            "44e96967-ab99-41ed-8b41-5bc70a5ca1a9",
            datetime(2015, 1, 1)
        ),
        (
            99999,
            "Test Product",
            "TEST-99999",
            False,
            False,
            "Blue",
            100,
            50,
            Decimal("10.0000"),
            Decimal("20.0000"),
            None,
            None,
            None,
            None,
            0,
            None,
            None,
            None,
            None,
            None,
            datetime(2026, 1, 1),
            None,
            None,
            "00000000-0000-0000-0000-000000000001",
            datetime(2026, 1, 1)
        )
    ],
    product_bronze_df.schema
)

display(product_merge_test_source)

In [ ]:
product_test_delta = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.product_merge_test"
)

(
    product_test_delta.alias("target")
    .merge(
        product_merge_test_source.alias("source"),
        "target.ProductID = source.ProductID"
    )
    .whenMatchedUpdateAll(
        condition="source.ModifiedDate > target.ModifiedDate"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [ ]:
display(
    spark.sql("""
        DESCRIBE HISTORY adb_adventureworks_dev.silver.product_merge_test
        LIMIT 1
    """).select(
        "version",
        "operation",
        "operationMetrics"
    )
)


In [ ]:
display(
    spark.table(
        "adb_adventureworks_dev.silver.product_merge_test"
    )
    .filter(F.col("ProductID").isin(940, 99999))
    .select(
        "ProductID",
        "Name",
        "ProductNumber",
        "ModifiedDate"
    )
    .orderBy("ProductID")
)

In [ ]:
product_merge_test_final = spark.table(
    "adb_adventureworks_dev.silver.product_merge_test"
)

print("Total rows:", product_merge_test_final.count())

print(
    "Distinct ProductIDs:",
    product_merge_test_final.select("ProductID").distinct().count()
)

In [ ]:
%sql
DROP TABLE adb_adventureworks_dev.silver.product_merge_test;

In [ ]:
customer_incremental_path = (
    bronze_path + "sales/customer/incremental/"
)

customer_incremental_df = (
    spark.read
    .format("parquet")
    .load(customer_incremental_path)
)

print("Total incremental rows:", customer_incremental_df.count())

print(
    "Distinct CustomerIDs:",
    customer_incremental_df.select("CustomerID").distinct().count()
)

display(
    customer_incremental_df
    .select("CustomerID", "ModifiedDate")
    .orderBy("CustomerID", "ModifiedDate")
)

In [ ]:
from pyspark.sql import functions as F

customer_file_counts = (
    customer_incremental_df
    .select(
        F.col("_metadata.file_path").alias("SourceFile")
    )
    .groupBy("SourceFile")
    .count()
    .orderBy("SourceFile")
)

display(customer_file_counts)

In [ ]:
small_customer_df = customer_incremental_df

display(
    small_customer_df
    .select("CustomerID", "AccountNumber", "ModifiedDate")
    .orderBy("CustomerID", "ModifiedDate")
)


In [ ]:
from pyspark.sql import functions as F

customer_columns = small_customer_df.columns

customer_version_check = (
    small_customer_df
    .groupBy("CustomerID", "ModifiedDate")
    .agg(
        F.count("*").alias("ExtractedCopies"),
        F.countDistinct(
            F.struct(*[F.col(c) for c in customer_columns])
        ).alias("DistinctFullRecords")
    )
    .orderBy("CustomerID", "ModifiedDate")
)

display(customer_version_check)

In [ ]:
customer_merge_source = (
    small_customer_df
    .dropDuplicates(["CustomerID", "ModifiedDate"])
)

print("Rows before deduplication:", small_customer_df.count())
print("Rows after deduplication:", customer_merge_source.count())
print(
    "Distinct CustomerIDs:",
    customer_merge_source.select("CustomerID").distinct().count()
)

display(
    customer_merge_source
    .select("CustomerID", "AccountNumber", "ModifiedDate")
    .orderBy("CustomerID")
)

In [ ]:
from pyspark.sql import functions as F

customer_silver_df = spark.table(
    "adb_adventureworks_dev.silver.customer"
)

customer_comparison = (
    customer_merge_source.alias("source")
    .join(
        customer_silver_df.alias("target"),
        on="CustomerID",
        how="left"
    )
    .select(
        F.col("source.CustomerID"),
        F.col("source.ModifiedDate").alias("SourceModifiedDate"),
        F.col("target.ModifiedDate").alias("SilverModifiedDate"),
        F.when(
            F.col("target.CustomerID").isNull(),
            "INSERT"
        ).when(
            F.col("source.ModifiedDate") >
            F.col("target.ModifiedDate"),
            "UPDATE"
        ).when(
            F.col("source.ModifiedDate") ==
            F.col("target.ModifiedDate"),
            "NO CHANGE"
        ).otherwise("OLDER SOURCE")
        .alias("MergeAction")
    )
    .orderBy("CustomerID")
)

display(customer_comparison)

In [ ]:
from delta.tables import DeltaTable

customer_delta_table = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.customer"
)

(
    customer_delta_table.alias("target")
    .merge(
        customer_merge_source.alias("source"),
        "target.CustomerID = source.CustomerID"
    )
    .whenMatchedUpdateAll(
        condition="source.ModifiedDate > target.ModifiedDate"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [ ]:
customer_history = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.customer"
).history(1)

display(
    customer_history.select(
        "version",
        "operation",
        "operationMetrics"
    )
)

In [ ]:
customer_final_df = spark.table(
    "adb_adventureworks_dev.silver.customer"
)

print("Total rows:", customer_final_df.count())
print(
    "Distinct CustomerIDs:",
    customer_final_df.select("CustomerID").distinct().count()
)
print(
    "Duplicate CustomerIDs:",
    customer_final_df.count()
    - customer_final_df.select("CustomerID").distinct().count()
)

In [ ]:
from pyspark.sql import functions as F

header_incremental_path = (
    bronze_path + "sales/salesorderheader/incremental/"
)

header_incremental_df = (
    spark.read
    .format("parquet")
    .load(header_incremental_path)
)

header_file_counts = (
    header_incremental_df
    .groupBy(F.col("_metadata.file_path").alias("FilePath"))
    .count()
    .orderBy("FilePath")
)

display(header_file_counts)

print("Total incremental rows:", header_incremental_df.count())

print(
    "Distinct SalesOrderIDs:",
    header_incremental_df
    .select("SalesOrderID")
    .distinct()
    .count()
)

In [ ]:
header_version_check = (
    header_incremental_df
    .groupBy("SalesOrderID", "ModifiedDate")
    .agg(
        F.count("*").alias("ExtractedCopies"),
        F.countDistinct(
            F.struct(*[
                F.col(c)
                for c in header_incremental_df.columns
                if c != "_metadata"
            ])
        ).alias("DistinctFullRecords")
    )
    .orderBy("SalesOrderID", "ModifiedDate")
)

display(header_version_check)

print(
    "Unique SalesOrderID and ModifiedDate combinations:",
    header_version_check.count()
)

print(
    "Conflicting versions:",
    header_version_check
    .filter(F.col("DistinctFullRecords") > 1)
    .count()
)

In [ ]:
header_merge_source = (
    header_incremental_df
    .dropDuplicates(["SalesOrderID", "ModifiedDate"])
)

print("Rows before deduplication:", header_incremental_df.count())
print("Rows after deduplication:", header_merge_source.count())

print(
    "Distinct SalesOrderIDs:",
    header_merge_source.select("SalesOrderID").distinct().count()
)

display(
    header_merge_source
    .select("SalesOrderID", "OrderDate", "CustomerID", "ModifiedDate")
    .orderBy("SalesOrderID")
)

In [ ]:
header_silver_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderheader"
)

header_comparison = (
    header_merge_source.alias("source")
    .join(
        header_silver_df.alias("target"),
        on="SalesOrderID",
        how="left"
    )
    .select(
        F.col("source.SalesOrderID"),
        F.col("source.ModifiedDate").alias("SourceModifiedDate"),
        F.col("target.ModifiedDate").alias("SilverModifiedDate"),
        F.when(
            F.col("target.SalesOrderID").isNull(),
            "INSERT"
        ).when(
            F.col("source.ModifiedDate") >
            F.col("target.ModifiedDate"),
            "UPDATE"
        ).when(
            F.col("source.ModifiedDate") ==
            F.col("target.ModifiedDate"),
            "NO CHANGE"
        ).otherwise("OLDER SOURCE")
        .alias("MergeAction")
    )
    .orderBy("SalesOrderID")
)

display(header_comparison)

print("Merge action counts:")
display(
    header_comparison
    .groupBy("MergeAction")
    .count()
    .orderBy("MergeAction")
)

In [ ]:
header_delta_table = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.salesorderheader"
)

(
    header_delta_table.alias("target")
    .merge(
        header_merge_source.alias("source"),
        "target.SalesOrderID = source.SalesOrderID"
    )
    .whenMatchedUpdateAll(
        condition="source.ModifiedDate > target.ModifiedDate"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [ ]:
header_history = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.salesorderheader"
).history(1)

display(
    header_history.select(
        "version",
        "operation",
        "operationMetrics"
    )
)

In [ ]:
header_final_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderheader"
)

print("Total rows:", header_final_df.count())

print(
    "Distinct SalesOrderIDs:",
    header_final_df.select("SalesOrderID").distinct().count()
)

print(
    "Duplicate SalesOrderIDs:",
    header_final_df.count()
    - header_final_df.select("SalesOrderID").distinct().count()
)

In [ ]:
from pyspark.sql import functions as F

detail_incremental_path = (
    "abfss://bronze@awdata2022.dfs.core.windows.net/"
    "sales/salesorderdetail/incremental/"
)

detail_incremental_df = (
    spark.read
    .parquet(detail_incremental_path)
    .withColumn(
        "SourceFile",
        F.col("_metadata.file_path")
    )
)

display(
    detail_incremental_df
    .groupBy("SourceFile")
    .agg(
        F.count("*").alias("RowCount"),
        F.countDistinct("SalesOrderID").alias("DistinctSalesOrderIDs"),
        F.countDistinct("SalesOrderDetailID").alias("DistinctDetailIDs"),
        F.min("ModifiedDate").alias("MinModifiedDate"),
        F.max("ModifiedDate").alias("MaxModifiedDate")
    )
    .orderBy("SourceFile")
)

In [ ]:
detail_version_check = (
    detail_incremental_df
    .groupBy(
        "SalesOrderID",
        "SalesOrderDetailID",
        "ModifiedDate"
    )
    .agg(
        F.count("*").alias("OccurrenceCount"),
        F.countDistinct(
            F.to_json(
                F.struct(*[
                    F.col(c)
                    for c in detail_incremental_df.columns
                    if c != "SourceFile"
                ])
            )
        ).alias("DistinctFullRecords")
    )
)

display(
    detail_version_check
    .groupBy("DistinctFullRecords")
    .agg(
        F.count("*").alias("CompositeKeyVersions")
    )
)

In [ ]:
detail_merge_source = (
    detail_incremental_df
    .dropDuplicates([
        "SalesOrderID",
        "SalesOrderDetailID",
        "ModifiedDate"
    ])
    .drop("SourceFile")
)

print("Rows before deduplication:", detail_incremental_df.count())
print("Rows after deduplication:", detail_merge_source.count())

print(
    "Distinct composite keys:",
    detail_merge_source
    .select("SalesOrderID", "SalesOrderDetailID")
    .distinct()
    .count()
)

In [ ]:
detail_silver_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

detail_comparison = (
    detail_merge_source.alias("source")
    .join(
        detail_silver_df.alias("target"),
        on=["SalesOrderID", "SalesOrderDetailID"],
        how="left"
    )
    .select(
        F.col("source.SalesOrderID"),
        F.col("source.SalesOrderDetailID"),
        F.col("source.ModifiedDate").alias("SourceModifiedDate"),
        F.col("target.ModifiedDate").alias("SilverModifiedDate"),
        F.when(
            F.col("target.SalesOrderID").isNull(),
            "INSERT"
        )
        .when(
            F.col("source.ModifiedDate") > F.col("target.ModifiedDate"),
            "UPDATE"
        )
        .when(
            F.col("source.ModifiedDate") == F.col("target.ModifiedDate"),
            "NO CHANGE"
        )
        .otherwise("OLDER SOURCE")
        .alias("MergeAction")
    )
)

display(
    detail_comparison
    .groupBy("MergeAction")
    .count()
)

In [ ]:
from delta.tables import DeltaTable

detail_delta_table = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.salesorderdetail"
)

(
    detail_delta_table.alias("target")
    .merge(
        detail_merge_source.alias("source"),
        """
        target.SalesOrderID = source.SalesOrderID
        AND target.SalesOrderDetailID = source.SalesOrderDetailID
        """
    )
    .whenMatchedUpdateAll(
        condition="source.ModifiedDate > target.ModifiedDate"
    )
    .whenNotMatchedInsertAll()
    .execute()
)

In [ ]:
detail_history = DeltaTable.forName(
    spark,
    "adb_adventureworks_dev.silver.salesorderdetail"
).history(1)

display(
    detail_history.select(
        "version",
        "operation",
        "operationMetrics"
    )
)

In [ ]:
detail_final_df = spark.table(
    "adb_adventureworks_dev.silver.salesorderdetail"
)

total_rows = detail_final_df.count()

distinct_keys = (
    detail_final_df
    .select("SalesOrderID", "SalesOrderDetailID")
    .distinct()
    .count()
)

print("Total rows:", total_rows)
print("Distinct composite keys:", distinct_keys)
print("Duplicate composite keys:", total_rows - distinct_keys)

In [ ]:
silver_tables = [
    "customer",
    "salesorderheader",
    "salesorderdetail",
    "product"
]

for table_name in silver_tables:
    print(f"\n{'=' * 60}")
    print(f"TABLE: {table_name}")
    print(f"{'=' * 60}")

    spark.table(
        f"adb_adventureworks_dev.silver.{table_name}"
    ).printSchema()

In [ ]:
from pyspark.sql import functions as F

customers = spark.table("adb_adventureworks_dev.silver.customer")
headers = spark.table("adb_adventureworks_dev.silver.salesorderheader")
details = spark.table("adb_adventureworks_dev.silver.salesorderdetail")
products = spark.table("adb_adventureworks_dev.silver.product")

detail_count = details.count()

checks = [
    (
        "Sales details without matching order",
        details.join(
            headers.select("SalesOrderID"),
            "SalesOrderID",
            "left_anti"
        ).count()
    ),
    (
        "Sales details without matching product",
        details.join(
            products.select("ProductID"),
            "ProductID",
            "left_anti"
        ).count()
    ),
    (
        "Orders without matching customer",
        headers.join(
            customers.select("CustomerID"),
            "CustomerID",
            "left_anti"
        ).count()
    ),
    (
        "Sales details with null SalesOrderID",
        details.filter(F.col("SalesOrderID").isNull()).count()
    ),
    (
        "Sales details with null ProductID",
        details.filter(F.col("ProductID").isNull()).count()
    )
]

for check_name, result in checks:
    print(f"{check_name}: {result}")

print(f"\nTotal sales detail rows: {detail_count}")

In [ ]:
headers.select(
    F.min("OrderDate").alias("FirstOrderDate"),
    F.max("OrderDate").alias("LastOrderDate"),
    F.countDistinct(
        F.to_date("OrderDate")
    ).alias("DistinctOrderDates")
).show(truncate=False)

In [ ]:
print(
    "Orders with null OrderDate:",
    headers.filter(F.col("OrderDate").isNull()).count()
)